# ECG-RAMBA Revision Runner

This notebook is a lightweight Colab runner for the reviewer-response pipeline. Keep `01_exploratory.ipynb` as an archive. Run scripts from the repository and write all reusable outputs to `reports/revision/`.

Configured Drive path: `/content/drive/MyDrive/ECG-Ramba`. The repo may live either directly in that folder or in `/content/drive/MyDrive/ECG-Ramba/ECG-RAMBA`.


## 1. Mount Drive And Enter Repo

Upload or clone the repository under `MyDrive/ECG-Ramba` before running this cell. Set `DRIVE_ROOT` below if your Drive folder is different.


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Not running in Colab or Drive already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_CANDIDATES = [DRIVE_ROOT / 'ECG-RAMBA', DRIVE_ROOT]
REPO_DIR = next((p for p in REPO_CANDIDATES if (p / 'configs' / 'config.py').exists()), None)
assert REPO_DIR is not None, f'Repo not found under: {REPO_CANDIDATES}'
os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
%cd {REPO_DIR}
print(f'Drive root: {DRIVE_ROOT}')
print(f'Repo dir  : {REPO_DIR}')
!pwd
!git status --short --branch || true


## 2. Install Lightweight Dependencies

This cell avoids forcing `mamba-ssm`. For full model training/inference on Colab, install the same Mamba wheels you used in the original exploratory notebook or run the old setup cell once.


In [ ]:
!python --version
!pip install -q numpy==1.26.4 pandas scipy scikit-learn tqdm wfdb joblib matplotlib seaborn packaging neurokit2 iterative-stratification thop


## 3. Check Drive Artifacts

The Drive root should contain `WFDB-ChapmanShaoxing.zip`, `PTB-XL.zip`, `Georgia.zip`, and `model/`. The patched config will use `model/` when it contains fold checkpoints and `models/` does not.


In [ ]:
from pathlib import Path

for candidate in [Path('models'), Path('model'), DRIVE_ROOT / 'model', DRIVE_ROOT / 'models']:
    fold_ckpts = sorted(candidate.glob('fold*_best.pt'))
    onnx_files = sorted(candidate.glob('fold*_best.onnx'))
    pca_file = candidate / 'global_pca_zeroshot.pkl'
    if candidate.exists():
        print(f'{candidate}: {len(fold_ckpts)} pt checkpoints, {len(onnx_files)} onnx files, PCA={pca_file.exists()}')

if not any(p.exists() and list(p.glob('fold*_best.pt')) for p in [Path('models'), Path('model'), DRIVE_ROOT / 'model', DRIVE_ROOT / 'models']):
    print('Missing fold*_best.pt. Upload/copy checkpoints into repo/model, repo/models, DRIVE_ROOT/model, or DRIVE_ROOT/models before full evaluation.')

for name in ['WFDB-ChapmanShaoxing.zip', 'PTB-XL.zip', 'Georgia.zip']:
    p = DRIVE_ROOT / name
    print(f'{name}: {p.exists()} ({p})')


## 4. A0 Protocol Audit

Run this before any expensive job. Do not use manuscript numbers until the warnings are understood or resolved.


In [ ]:
!python scripts/revision/00_audit_protocol.py


In [ ]:
import json
from pathlib import Path

audit_path = Path('reports/revision/audit_protocol.json')
audit = json.loads(audit_path.read_text(encoding='utf-8'))
audit['warnings']


## 5. Main Revision Jobs

Uncomment one command at a time. Prefer generating reusable `.npz` prediction files, then run calibration/CI from those files.


In [ ]:
# Existing repo scripts, if models/data are ready:
# !python scripts/eval_oof.py
# !python scripts/eval_zeroshot.py

# Future revision scripts should write:
# reports/revision/oof_predictions.npz
# reports/revision/ptbxl_predictions.npz
# reports/revision/cpsc_predictions.npz


## 6. Calibration And Bootstrap CI

Prediction NPZ files must contain `y_true` and `y_prob` arrays with matching shape `(N, C)`.


In [ ]:
PREDICTION_FILE = Path('reports/revision/oof_predictions.npz')
if PREDICTION_FILE.exists():
    !python scripts/revision/04_calibration_ci.py --predictions {PREDICTION_FILE} --n-boot 1000
else:
    print(f'Missing prediction file: {PREDICTION_FILE}')
    print('Create it from OOF/zero-shot evaluation before running calibration/CI.')


## 7. Result Inventory

Use this cell after each run to confirm the artifact contract for the rebuttal/manuscript.


In [ ]:
!find reports/revision -maxdepth 3 -type f | sort || true


## Notes

- Keep CPSC labels/order canonical across all CPSC cells/scripts.
- Use the PTB superclass mapping from `scripts.revision.common.PTB_SUPERCLASS_MAPPING`.
- Do not describe HRV36 as full HRV unless new features are implemented and the model is retrained.
- Treat inference ablation as diagnostic; reviewer-facing component claims need protocol-fair baselines.
